# IMDB Sentiment Classification — ECE 364 Project Option #2

## Approach

We use four techniques to classify movie reviews as positive or negative:

1. **Small pretrained BERT** (`google/bert_uncased_L-12_H-128_A-2`, ~7.4M params) — a 12-layer
   transformer pretrained on Wikipedia and BooksCorpus. Deeper than it looks: 12 layers with a
   narrow hidden size of 128 stays under the 10M parameter cap.

2. **MLM domain adaptation** — before fine-tuning on sentiment labels, we continue training the
   model on raw IMDB review text using Masked Language Modeling (randomly mask 15% of tokens,
   predict them back). No labels needed. This teaches the model IMDB-specific vocabulary and
   writing style.

3. **Knowledge Distillation (KD)** — instead of training on hard labels (0 or 1) alone, we also
   train the student to mimic a large teacher model's soft probability outputs. The teacher
   (`textattack/bert-base-uncased-imdb`, 110M params, ~94% accuracy) provides richer training
   signal than binary labels alone.

4. **Semi-supervised KD on unlabeled data** — we run the teacher over the unlabeled test set
   AND the extra aclImdb unsup reviews, cache those soft predictions, and include them in
   training with a KD-only loss (no ground-truth label needed). This exposes the student to the
   exact distribution it will be tested on.


## 0. Setup

In [ ]:
import os, gc, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel, SWALR
from transformers import (
    AutoTokenizer, AutoConfig,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 2026
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    print("GPU:", GPU_NAME)
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    DTYPE = torch.bfloat16 if any(x in GPU_NAME for x in
        ["A100","A10","L4","H100","H200","Blackwell","RTX PRO"]) else torch.float16
else:
    DTYPE = torch.float16
print("AMP dtype:", DTYPE)

In [ ]:
# ── Config ──
# Folder structure expected:
#   imdb_sentiment_local.ipynb   ← this notebook
#   imdb_project/
#       train.csv
#       test.csv
#       aclImdb_unsup.csv        ← optional
PROJECT_DIR     = "imdb_project"
TRAIN_CSV       = f"{PROJECT_DIR}/train.csv"
TEST_CSV        = f"{PROJECT_DIR}/test.csv"
ACL_UNSUP_CSV   = f"{PROJECT_DIR}/aclImdb_unsup.csv"
WORK_DIR        = f"{PROJECT_DIR}/work_dir_simple"
PREDICTIONS_CSV = f"{PROJECT_DIR}/predictions_simple.csv"
os.makedirs(WORK_DIR, exist_ok=True)

# Student: 12 layers x hidden 128 x 2 heads = ~7.4M params
STUDENT_NAME = "google/bert_uncased_L-12_H-128_A-2"

# Teacher: BERT-base fine-tuned on IMDB (~94% accuracy, 110M params)
# Used only during training to generate soft labels — NOT in the final deliverable
TEACHER_NAME = "textattack/bert-base-uncased-imdb"

# Sequence
MAX_LEN  = 512
HEAD_LEN = 128
TAIL_LEN = MAX_LEN - HEAD_LEN - 2  # 382

# MLM pretraining
MLM_EPOCHS     = 12
MLM_BATCH_SIZE = 128
MLM_SEQ_LEN    = 256
MLM_LR         = 3e-4

# Fine-tuning
FT_EPOCHS     = 16
FT_BATCH_SIZE = 128
FT_EVAL_BATCH = 256
FT_LR         = 3e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.10
GRAD_ACCUM    = 1       # effective batch = 128

# KD hyperparameters
KD_ALPHA = 0.7    # fraction of loss from KD (vs hard labels); high because teacher is strong
KD_T     = 2.0    # temperature: softens probabilities to make KD signal richer

# SWA — averages model weights over the last few epochs
SWA_START_EPOCH = 13   # start averaging from this epoch (1-indexed)
SWA_LR          = 1e-5

LABEL2ID = {"negative": 0, "positive": 1}
ID2LABEL  = {0: "negative", 1: "positive"}
print("Config ready. WORK_DIR:", WORK_DIR)

## 1. Load data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
print("Train:", train_df.shape, "| Test:", test_df.shape)
print(train_df["label"].value_counts())

train_df["label_id"] = train_df["label"].str.lower().map(LABEL2ID)
assert train_df["label_id"].isna().sum() == 0, "Unexpected label values"

lens = train_df["review"].str.split().str.len()
print(f"\nReview length — mean: {lens.mean():.0f}, median: {lens.median():.0f}, "
      f"max: {lens.max()}, >512 words: {(lens>512).mean()*100:.1f}%")

## 2. Train / val split (90 / 10, stratified)

In [ ]:
trn_df, val_df = train_test_split(
    train_df, test_size=0.10, stratify=train_df["label_id"], random_state=SEED
)
trn_df = trn_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(trn_df):,} | Val: {len(val_df):,}")
print("Train labels:", trn_df["label_id"].value_counts().to_dict())

## 3. Verify student model is under 10M params

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME)
cfg = AutoConfig.from_pretrained(
    STUDENT_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
)
_probe = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, config=cfg)
total = sum(p.numel() for p in _probe.parameters())
print(f"Student total params: {total:>12,}")
print(f"10M limit:            {10_000_000:>12,}")
assert total < 10_000_000, f"Over limit! ({total:,})"
print("✓ Under 10M\n")
print("Param breakdown:")
for name, mod in _probe.named_children():
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {name:30s} {n:>10,}")
del _probe; gc.collect()

## 4. Head + tail tokenization

BERT has a 512-token limit. For longer reviews we keep the **first 128 tokens**
(where the reviewer sets context) and the **last 382 tokens** (where they usually
state their verdict). This beats simply cutting at 512.

In [ ]:
def head_tail_tokenize(text):
    cls = tokenizer.cls_token_id
    sep = tokenizer.sep_token_id
    pad = tokenizer.pad_token_id
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    if len(ids) <= MAX_LEN - 2:
        tokens = [cls] + ids + [sep]
    else:
        tokens = [cls] + ids[:HEAD_LEN] + ids[-TAIL_LEN:] + [sep]
    attn  = [1] * len(tokens)
    pad_n = MAX_LEN - len(tokens)
    tokens = (tokens + [pad] * pad_n)[:MAX_LEN]
    attn   = (attn   + [0]   * pad_n)[:MAX_LEN]
    return {"input_ids": tokens, "attention_mask": attn}

## 5. Phase 1 — MLM domain adaptation

We fine-tune the student's weights on raw IMDB text using **Masked Language Modeling**:
randomly replace 15% of tokens with `[MASK]` and train the model to predict the
originals. No sentiment labels are used here — this is purely unsupervised.

**Why it helps:** the model arrives pre-trained on Wikipedia/books. Movie reviews use
very different vocabulary ("cinematography", "plotline", "Stallone"). A few MLM epochs
on IMDB text adapts the representations before we ever touch sentiment labels.

Corpus: labeled train reviews + val reviews + test reviews + aclImdb unsup reviews (~100K total).

In [ ]:
class MLMDataset(Dataset):
    def __init__(self, texts, tok, max_len):
        self.texts = list(texts); self.tok = tok; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in enc.items()}

# Collect all available text (labels never used)
corpus = pd.concat(
    [trn_df["review"], val_df["review"], test_df["review"]], ignore_index=True
).tolist()

if os.path.exists(ACL_UNSUP_CSV):
    extra = pd.read_csv(ACL_UNSUP_CSV)["review"].dropna().astype(str).tolist()
    corpus += extra
    print(f"MLM corpus: {len(corpus):,} reviews (including {len(extra):,} aclImdb unsup)")
else:
    print(f"MLM corpus: {len(corpus):,} reviews")

mlm_ds       = MLMDataset(corpus, tokenizer, max_len=MLM_SEQ_LEN)
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
mlm_loader   = DataLoader(mlm_ds, batch_size=MLM_BATCH_SIZE, shuffle=True,
                          collate_fn=mlm_collator, num_workers=0, pin_memory=True)
print(f"MLM loader: {len(mlm_loader):,} batches/epoch")

In [ ]:
MLM_SAVE = os.path.join(WORK_DIR, "student_mlm")

if os.path.exists(os.path.join(MLM_SAVE, "config.json")):
    print(f"✓ MLM checkpoint already exists at {MLM_SAVE}")
    print("  Delete that folder to redo MLM from scratch.")
else:
    print("Starting MLM pretraining...")
    mlm_model = AutoModelForMaskedLM.from_pretrained(STUDENT_NAME).to(DEVICE)
    optim = torch.optim.AdamW(mlm_model.parameters(), lr=MLM_LR, weight_decay=0.01)
    total_steps = len(mlm_loader) * MLM_EPOCHS
    sched  = get_cosine_schedule_with_warmup(optim, int(0.05*total_steps), total_steps)
    scaler = GradScaler("cuda")

    for epoch in range(MLM_EPOCHS):
        mlm_model.train()
        running, n = 0.0, 0
        pbar = tqdm(mlm_loader, desc=f"MLM epoch {epoch+1}/{MLM_EPOCHS}")
        for batch in pbar:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            optim.zero_grad(set_to_none=True)
            with autocast("cuda", dtype=DTYPE):
                loss = mlm_model(**batch).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
            scaler.step(optim); scaler.update(); sched.step()
            running += loss.item(); n += 1
            if n % 100 == 0:
                pbar.set_postfix(loss=f"{running/n:.4f}")
        print(f"  Epoch {epoch+1}: mean loss = {running/n:.4f}")

    mlm_model.save_pretrained(MLM_SAVE)
    tokenizer.save_pretrained(MLM_SAVE)
    print(f"Saved MLM-pretrained model to {MLM_SAVE}")
    del mlm_model, optim, sched, scaler
    gc.collect(); torch.cuda.empty_cache()

## 6. Phase 2 — Cache teacher soft labels

We load a large BERT-base model already fine-tuned on IMDB and run it over:
- The **labeled train set** → used for KD during fine-tuning
- The **labeled val set** → used for checking teacher label order
- The **unlabeled set** (test + aclImdb unsup) → used for semi-supervised KD

We cache all logits to disk so the teacher only runs once.
The teacher is then deleted from GPU — it is never used at inference time.

In [ ]:
@torch.no_grad()
def get_logits(model, tokenizer, texts, batch_size=16):
    model.eval()
    all_logits = []
    for i in tqdm(range(0, len(texts), batch_size), desc="  Teacher inference"):
        batch = tokenizer(texts[i:i+batch_size], padding=True, truncation=True,
                          max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with autocast("cuda", dtype=DTYPE):
            out = model(**batch)
        all_logits.append(out.logits.float().cpu())
    return torch.cat(all_logits, dim=0)

TRN_LOGITS_PATH   = os.path.join(WORK_DIR, "teacher_trn_logits.pt")
VAL_LOGITS_PATH   = os.path.join(WORK_DIR, "teacher_val_logits.pt")
UNSUP_LOGITS_PATH = os.path.join(WORK_DIR, "teacher_unsup_logits.pt")

# Build unlabeled set: test reviews + aclImdb unsup reviews
unsup_texts = test_df["review"].tolist()
if os.path.exists(ACL_UNSUP_CSV):
    extra_df = pd.read_csv(ACL_UNSUP_CSV)
    unsup_texts += extra_df["review"].dropna().astype(str).tolist()
print(f"Unlabeled KD set size: {len(unsup_texts):,}")

In [ ]:
# Load cached logits if they exist, otherwise run teacher inference
if all(os.path.exists(p) for p in [TRN_LOGITS_PATH, VAL_LOGITS_PATH, UNSUP_LOGITS_PATH]):
    print("✓ Loading cached teacher logits from disk...")
    teacher_trn_logits   = torch.load(TRN_LOGITS_PATH,   map_location="cpu")
    teacher_val_logits   = torch.load(VAL_LOGITS_PATH,   map_location="cpu")
    teacher_unsup_logits = torch.load(UNSUP_LOGITS_PATH, map_location="cpu")
else:
    print(f"Loading teacher: {TEACHER_NAME}")
    ttok  = AutoTokenizer.from_pretrained(TEACHER_NAME)
    tmod  = AutoModelForSequenceClassification.from_pretrained(TEACHER_NAME).to(DEVICE)
    n_teacher = sum(p.numel() for p in tmod.parameters())
    print(f"Teacher params: {n_teacher:,}  (training only — not in deliverable)")

    print("\nRunning teacher on train set...")
    teacher_trn_logits   = get_logits(tmod, ttok, trn_df["review"].tolist())
    print("Running teacher on val set...")
    teacher_val_logits   = get_logits(tmod, ttok, val_df["review"].tolist())
    print("Running teacher on unlabeled set...")
    teacher_unsup_logits = get_logits(tmod, ttok, unsup_texts)

    torch.save(teacher_trn_logits,   TRN_LOGITS_PATH)
    torch.save(teacher_val_logits,   VAL_LOGITS_PATH)
    torch.save(teacher_unsup_logits, UNSUP_LOGITS_PATH)
    print("Saved all teacher logits to disk.")

    del tmod, ttok
    gc.collect(); torch.cuda.empty_cache()

print(f"\nLogit shapes — train: {teacher_trn_logits.shape}, "
      f"val: {teacher_val_logits.shape}, unsup: {teacher_unsup_logits.shape}")

In [ ]:
# Verify teacher label ordering matches ours (negative=0, positive=1)
val_labels = val_df["label_id"].values
pred        = teacher_val_logits.argmax(-1).numpy()
acc_as_is   = (pred == val_labels).mean()
acc_flipped = ((1 - pred) == val_labels).mean()
print(f"Teacher val acc as-is:   {acc_as_is:.4f}")
print(f"Teacher val acc flipped: {acc_flipped:.4f}")

if acc_flipped > acc_as_is:
    print("→ Flipping teacher logits to match our label order")
    teacher_trn_logits   = teacher_trn_logits[:, [1, 0]]
    teacher_val_logits   = teacher_val_logits[:, [1, 0]]
    teacher_unsup_logits = teacher_unsup_logits[:, [1, 0]]

final_acc = (teacher_val_logits.argmax(-1).numpy() == val_labels).mean()
print(f"Teacher val acc (final): {final_acc:.4f}")

## 7. Datasets and DataLoaders

Three datasets:
- **Labeled train** — has review text, hard label, AND teacher soft logits → full KD loss
- **Unlabeled (semi-sup)** — has review text and teacher soft logits, NO hard label → KD-only loss
- **Val / Test** — review text (+ hard label for val) → evaluation / final inference

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels=None, teacher_logits=None):
        self.texts           = list(texts)
        self.labels          = labels           # None for unlabeled
        self.teacher_logits  = teacher_logits   # None for val/test
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = head_tail_tokenize(self.texts[i])
        item = {
            "input_ids":      torch.tensor(enc["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[i]), dtype=torch.long)
        if self.teacher_logits is not None:
            item["teacher_logits"] = self.teacher_logits[i].clone()
        return item

trn_ds      = IMDBDataset(trn_df["review"],  labels=trn_df["label_id"].values,
                          teacher_logits=teacher_trn_logits)
val_ds      = IMDBDataset(val_df["review"],  labels=val_df["label_id"].values)
unsup_ds    = IMDBDataset(unsup_texts,        teacher_logits=teacher_unsup_logits)
test_ds     = IMDBDataset(test_df["review"])

trn_loader   = DataLoader(trn_ds,   batch_size=FT_BATCH_SIZE,  shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=FT_EVAL_BATCH,  shuffle=False, num_workers=0, pin_memory=True)
unsup_loader = DataLoader(unsup_ds, batch_size=FT_BATCH_SIZE,  shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=FT_EVAL_BATCH,  shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(trn_loader)} | Unsup batches: {len(unsup_loader)}")

## 8. Phase 3 — Fine-tuning with Knowledge Distillation + SWA

**Loss function for labeled batches:**

$$\mathcal{L}_{\text{labeled}} = \alpha \cdot T^2 \cdot \mathrm{KL}\!\left(\frac{s}{T}\,\Big\|\,\frac{t}{T}\right) + (1-\alpha)\cdot\mathrm{CE}(s,\,y)$$

- $s$ = student logits, $t$ = teacher logits, $y$ = true label
- $\alpha = 0.7$: rely more on teacher (it's at 94%) than hard labels
- $T = 2.0$: temperature softens probabilities, giving the student more to learn

**Loss function for unlabeled batches (semi-supervised KD):**

$$\mathcal{L}_{\text{unlabeled}} = T^2 \cdot \mathrm{KL}\!\left(\frac{s}{T}\,\Big\|\,\frac{t}{T}\right)$$

**SWA (Stochastic Weight Averaging):** from epoch `SWA_START_EPOCH` onward, we keep a running
average of the model weights after each epoch. The averaged model typically sits in a flatter,
more generalizable region of the loss landscape than any single checkpoint.

In [ ]:
def kd_loss(student_logits, teacher_logits, T=KD_T):
    """KL divergence between temperature-scaled student and teacher distributions."""
    s = F.log_softmax(student_logits / T, dim=-1)
    t = F.softmax(teacher_logits    / T, dim=-1)
    return F.kl_div(s, t, reduction="batchmean") * (T * T)

def labeled_loss(s_logits, t_logits, labels):
    """KD loss + cross-entropy on hard labels."""
    kd = kd_loss(s_logits, t_logits)
    ce = F.cross_entropy(s_logits, labels)
    return KD_ALPHA * kd + (1 - KD_ALPHA) * ce

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        with autocast("cuda", dtype=DTYPE):
            out = model(input_ids=ids, attention_mask=mask)
        correct += (out.logits.argmax(-1) == labels).sum().item()
        total   += labels.size(0)
    return correct / total

In [ ]:
# Load student from MLM-pretrained checkpoint + add classification head
student = AutoModelForSequenceClassification.from_pretrained(
    MLM_SAVE, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID,
).to(DEVICE)

n_total = sum(p.numel() for p in student.parameters())
print(f"Student total params: {n_total:,} ({'✓ under 10M' if n_total < 10_000_000 else 'OVER LIMIT'})")

# AdamW with no weight decay on bias / LayerNorm
no_decay = ["bias", "LayerNorm.weight"]
grouped  = [
    {"params": [p for n,p in student.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n,p in student.named_parameters() if     any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optim = torch.optim.AdamW(grouped, lr=FT_LR)

total_steps = (len(trn_loader) // GRAD_ACCUM) * FT_EPOCHS
sched  = get_cosine_schedule_with_warmup(optim, int(WARMUP_RATIO * total_steps), total_steps)
scaler = GradScaler("cuda")

# SWA: keeps a running average of model weights from SWA_START_EPOCH onward
swa_model  = AveragedModel(student)
swa_sched  = SWALR(optim, swa_lr=SWA_LR)
swa_active = False
print(f"Training for {FT_EPOCHS} epochs, {total_steps} optimizer steps.")
print(f"SWA activates at epoch {SWA_START_EPOCH}.")

In [ ]:
best_val_acc = 0.0
BEST_CKPT   = os.path.join(WORK_DIR, "student_best")
log = []

for epoch in range(FT_EPOCHS):
    student.train()
    running_labeled, running_unsup, n = 0.0, 0.0, 0
    unsup_iter = iter(unsup_loader)
    optim.zero_grad(set_to_none=True)

    pbar = tqdm(trn_loader, desc=f"FT epoch {epoch+1}/{FT_EPOCHS}")
    for step, tb in enumerate(pbar):
        # ── labeled batch ──
        ids   = tb["input_ids"].to(DEVICE, non_blocking=True)
        mask  = tb["attention_mask"].to(DEVICE, non_blocking=True)
        lab   = tb["labels"].to(DEVICE, non_blocking=True)
        t_lg  = tb["teacher_logits"].to(DEVICE, non_blocking=True)
        with autocast("cuda", dtype=DTYPE):
            s_out = student(input_ids=ids, attention_mask=mask)
            l_lab = labeled_loss(s_out.logits.float(), t_lg.float(), lab)

        # ── unlabeled batch (KD only, no hard labels) ──
        try:
            ub = next(unsup_iter)
        except StopIteration:
            unsup_iter = iter(unsup_loader); ub = next(unsup_iter)
        u_ids  = ub["input_ids"].to(DEVICE, non_blocking=True)
        u_mask = ub["attention_mask"].to(DEVICE, non_blocking=True)
        u_tlg  = ub["teacher_logits"].to(DEVICE, non_blocking=True)
        with autocast("cuda", dtype=DTYPE):
            u_out = student(input_ids=u_ids, attention_mask=u_mask)
            l_uns = kd_loss(u_out.logits.float(), u_tlg.float())

        # Combined loss + gradient accumulation
        loss = (l_lab + l_uns) / GRAD_ACCUM
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(trn_loader):
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optim); scaler.update(); sched.step()
            optim.zero_grad(set_to_none=True)

        running_labeled += l_lab.item(); running_unsup += l_uns.item(); n += 1
        if n % 50 == 0:
            pbar.set_postfix(L_lab=f"{running_labeled/n:.4f}", L_uns=f"{running_unsup/n:.4f}")

    val_acc = evaluate(student, val_loader)
    log.append({"epoch": epoch+1, "L_labeled": running_labeled/n,
                "L_unsup": running_unsup/n, "val_acc": val_acc})
    print(f"  → epoch {epoch+1}: L_labeled={running_labeled/n:.4f}  "
          f"L_unsup={running_unsup/n:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        student.save_pretrained(BEST_CKPT)
        tokenizer.save_pretrained(BEST_CKPT)
        print(f"  ✓ New best: {val_acc:.4f} — saved to {BEST_CKPT}")

    # SWA: start averaging weights after SWA_START_EPOCH
    if (epoch + 1) >= SWA_START_EPOCH:
        if not swa_active:
            print(f"  → SWA activated at epoch {epoch+1}")
            swa_active = True
        swa_model.update_parameters(student)
        swa_sched.step()

print(f"\nBest single-checkpoint val accuracy: {best_val_acc:.4f}")
pd.DataFrame(log).to_csv(os.path.join(WORK_DIR, "training_log.csv"), index=False)

## 9. Compare best checkpoint vs SWA model, pick winner

In [ ]:
# Evaluate best single checkpoint
best_model = AutoModelForSequenceClassification.from_pretrained(BEST_CKPT).to(DEVICE)
acc_best = evaluate(best_model, val_loader)
print(f"Best checkpoint val acc: {acc_best:.4f}")

# Evaluate SWA model
swa_model.eval()
acc_swa = evaluate(swa_model, val_loader)
print(f"SWA model val acc:       {acc_swa:.4f}")

# Use whichever is better
if acc_swa >= acc_best:
    print("\n→ Using SWA model for predictions")
    final_model = swa_model
    final_acc   = acc_swa
else:
    print("\n→ Using best checkpoint for predictions")
    final_model = best_model
    final_acc   = acc_best

n_total = sum(p.numel() for p in (
    final_model.module if isinstance(final_model, AveragedModel) else final_model
).parameters())
print(f"Final val accuracy: {final_acc:.4f}")
print(f"Total params: {n_total:,}  ({'✓ under 10M' if n_total < 10_000_000 else 'OVER LIMIT'})")

## 10. Generate predictions.csv

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    for batch in tqdm(loader, desc="Predicting"):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        with autocast("cuda", dtype=DTYPE):
            out = model(input_ids=ids, attention_mask=mask)
        preds.append(out.logits.argmax(-1).cpu().numpy())
    return np.concatenate(preds)

import numpy as np
test_preds = predict(final_model, test_loader)
print("Prediction distribution:", pd.Series(test_preds).value_counts().to_dict())

out_df = pd.DataFrame({
    "Id":    test_df["Id"].values,
    "Label": [ID2LABEL[int(p)] for p in test_preds],
})
out_df.to_csv(PREDICTIONS_CSV, index=False)
print(f"\nWrote: {PREDICTIONS_CSV}")
print(out_df.head())

# Format checks
assert list(out_df.columns) == ["Id", "Label"], "Wrong columns"
assert set(out_df["Label"].unique()) <= {"positive", "negative"}, "Labels not lowercase"
assert len(out_df) == len(test_df), "Wrong row count"
print("\n✓ predictions.csv verified")

## 11. Summary for report

**Architecture:** `google/bert_uncased_L-12_H-128_A-2`
- 12 transformer layers, hidden size 128, 2 attention heads
- ~7.4M total parameters (under 10M limit)

**Technique 1 — MLM Domain Adaptation:**
- Continued pretraining on ~100K IMDB reviews using masked language modeling
- 5 epochs, seq-len 256, LR 3e-4, 15% masking probability
- No labels used — purely unsupervised

**Technique 2 — Knowledge Distillation:**
- Teacher: `textattack/bert-base-uncased-imdb` (BERT-base, 110M params, ~94% on IMDB)
- Loss: α × KL(student||teacher) + (1−α) × CE(student, hard label), α=0.7, T=2.0
- Teacher used only during training, not in the deliverable

**Technique 3 — Semi-supervised KD:**
- Teacher also run over 5,000 unlabeled test reviews + 49,507 aclImdb unsup reviews
- Student trained to match teacher on those too (KD loss only, no hard labels)
- Exposes student to the test distribution during training

**Optimizer:** AdamW, LR 3e-5, cosine schedule, warmup 10%, weight decay 0.01,
gradient clipping 1.0, mixed precision (bfloat16), effective batch size 128.
SWA activates at epoch 12, averaging weights over the final epochs into the delivered model.
